# Notebook 02: Train the LSTM detector and choose its threshold

The key idea from Phase 1 is straightforward:

1. Train an LSTM on healthy windows.
2. Ask it to reconstruct each input window.
3. Compare reconstructed and original readings to obtain one error score per window.
4. Evaluate candidate thresholds with precision, recall, F1 and accuracy.

**Important:** The threshold is chosen using *validation* cycles, not test cycles.
Run `main.py` for the entire workflow and to save trained models to disk.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Works if PyCharm runs notebooks from either the project folder or notebooks/.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

from data import load_windows

DATA_FILE = PROJECT / "data" / "PS1_demo.csv"
DATA_FORMAT = "cycles"  # change to "windows" for prepared 60-column Excel files

from sklearn.model_selection import train_test_split
from detection import train_autoencoder, reconstruction_error, choose_threshold
from faults import make_detection_examples

## 1. Load data and keep cycles separate

In [ ]:
windows, cycle_ids = load_windows(DATA_FILE, data_format=DATA_FORMAT, windows_per_cycle=3)
train_cycles, remaining_cycles = train_test_split(
    np.unique(cycle_ids), test_size=0.30, random_state=42
)
validation_cycles, test_cycles = train_test_split(
    remaining_cycles, test_size=0.50, random_state=42
)
healthy_train = windows[np.isin(cycle_ids, train_cycles)]
healthy_val = windows[np.isin(cycle_ids, validation_cycles)]
healthy_test = windows[np.isin(cycle_ids, test_cycles)]

print("Healthy train:", healthy_train.shape)
print("Healthy validation:", healthy_val.shape)
print("Held-out test (not used below):", healthy_test.shape)

## 2. Train the LSTM autoencoder

In [ ]:
model, scaler, history = train_autoencoder(
    healthy_train, healthy_val, epochs=12
)

## 3. Show the loss curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training MSE")
plt.plot(history.history["val_loss"], label="Validation MSE")
plt.xlabel("Epoch")
plt.ylabel("Mean squared reconstruction error")
plt.legend()
plt.show()

## 4. Generate validation faults, reconstruct the windows

In [ ]:
val_windows, val_labels = make_detection_examples(healthy_val)

val_scaled = scaler.transform(
    val_windows.reshape(-1, 1)
).reshape(val_windows.shape)

predicted = model.predict(val_scaled[..., None], verbose=0)
scores = reconstruction_error(val_scaled, predicted, method="mae")
print("One reconstruction error per window:", scores.shape)

## 5. Find the threshold from validation metrics

In [ ]:
threshold, table = choose_threshold(scores, val_labels)
print("Chosen threshold:", threshold)
print(table.sort_values("f1", ascending=False).head(5).to_string(index=False))

## 6. Plot precision, recall, F1 and accuracy against the threshold

In [ ]:
plt.figure(figsize=(9, 5))
for name in ["precision", "recall", "f1", "accuracy"]:
    plt.plot(table["threshold"], table[name], label=name)
plt.axvline(threshold, color="black", linestyle="--", label="Chosen threshold")
plt.xlabel("Reconstruction-error threshold")
plt.ylabel("Metric (0 to 1)")
plt.title("How the detection metrics change with threshold")
plt.legend()
plt.show()

### Important distinction from the original research

This learning version **automatically chooses the threshold with the highest F1**.
Your original approach involved inspecting how several metric curves relate.
This notebook makes those curves available for your own analysis;
the automatic F1 rule is a clearly labeled simplification.